# RLS — Gestão de acesso ao modelo semântico

Este notebook tem duas responsabilidades:
1. **Diagnóstico:** inspecionar `dim_funcionarios` para mapear colunas → escopos RLS
2. **Construção:** ler as listas SharePoint (rls_grupos, rls_overrides) e gerar `dim_rls_usuarios` no `lake_gold_fatos`

## Regras de derivação

### escopo
| critério | escopo |
|---|---|
| `cargo` ∋ "GERENTE" **AND** `secao` ∋ "SEDE" | `GERENTE_SEDE` |
| `secao` ∋ "SEDE" (demais cargos) | `ADMIN_CENTRAL` |
| demais (incluindo gerente de unidade) | `UNIDADE` |

### perfil_acesso
| critério | perfil |
|---|---|
| `cargo` ∋ "GERENTE" | `GERENTE` |
| `cargo` ∋ "COORD" | `COORDENADOR` |
| `cargo` ∋ "SUPERV" | `SUPERVISOR` |
| `secao` ∋ "PROG" AND escopo = UNIDADE | `PROGRAMADOR` |
| `secao` ∋ "SEDE" AND `cargo` ∋ "TÉCNIC" ou "ESPECIAL" | `ASSISTENTE` |
| SharePoint `rls_grupos` | `DADOS_GEDES` / `REPRESENTANTE` |
| fallback | `GERAL` |

### Listas SharePoint
- **`rls_grupos`**: atribuição manual de `DADOS_GEDES` e `REPRESENTANTE` (email, perfil_acesso, escopo, unidade, gerencias)
- **`rls_overrides`**: exceções individuais (email, escopo, perfil_acesso, unidade, gerencias, motivo, ativo)

## 1. Conexão Fabric

In [ ]:
import struct, pyodbc, pandas as pd

SQL_ENDPOINT = (
    'beu5bmmdbuwedpv62ucm524jzi-dmrv7k3fbwbevh5d4sidg3urfq'
    '.datawarehouse.fabric.microsoft.com'
)

from azure.identity import DeviceCodeCredential
_cred  = DeviceCodeCredential()
_token = _cred.get_token('https://database.windows.net/.default').token

_tb = _token.encode('utf-16-le')
_ts = struct.pack(f'<I{len(_tb)}s', len(_tb), _tb)

conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SQL_ENDPOINT};Encrypt=Yes;',
    attrs_before={1256: _ts},
)
print('Conectado.')

## 2. Inspecionar dim_funcionarios

In [ ]:
dim_func_df = pd.read_sql(
    'SELECT * FROM lake_gold_fatos.dbo.dim_funcionarios',
    conn
)
print(f'{len(dim_func_df):,} registros  |  {dim_func_df.columns.tolist()}')
dim_func_df.dtypes

In [ ]:
dim_func_df.head(10)

### 2a. Distribuições relevantes para o RLS
Preencher os nomes de coluna após ver o schema acima.

In [ ]:
# Ajustar os nomes de coluna conforme o schema real
# COL_CARGO   = 'cargo'      # coluna que indica função/cargo
# COL_LOTACAO = 'lotacao'    # coluna que indica UO ou sede
# COL_EMAIL   = 'email'

# Exemplo — descomentar e ajustar:
# print('=== Cargos distintos ===')
# print(dim_func_df[COL_CARGO].value_counts().to_string())
# print('\n=== Lotações distintas ===')
# print(dim_func_df[COL_LOTACAO].value_counts().to_string())

In [ ]:
# Analisar coluna 'secao': valores pipe-separados → quais unidades têm "SEDE"
secao_exploded = (
    dim_func_df['secao']
    .dropna()
    .str.split('|')
    .explode()
    .str.strip()
)

print('=== Todos os valores distintos em secao ===')
print(sorted(secao_exploded.unique()))

print('\n=== Valores que contêm "SEDE" ===')
sede_vals = secao_exploded[secao_exploded.str.contains('SEDE', case=False, na=False)]
print(sorted(sede_vals.unique()))

print('\n=== Funcionários cuja secao contém "SEDE" ===')
mask = dim_func_df['secao'].str.contains('SEDE', case=False, na=False)
print(f'{mask.sum()} registros')
dim_func_df[mask][['email', 'secao']].drop_duplicates().sort_values('secao')

In [ ]:
# Perfis relevantes para RLS: cargo (gerente/coordenador) e secao (programação)

print('=== Valores distintos em cargo ===')
print(sorted(dim_func_df['cargo'].dropna().unique()))

print('\n--- Contagens ---')

gerentes = dim_func_df['cargo'].str.contains('GERENTE', case=False, na=False)
coordenadores = dim_func_df['cargo'].str.contains('COORD', case=False, na=False)
programacao = dim_func_df['secao'].str.contains('PROG', case=False, na=False)

print(f'Gerentes      (cargo ∋ "GERENTE"):    {gerentes.sum():>4}')
print(f'Coordenadores (cargo ∋ "COORD"):      {coordenadores.sum():>4}')
print(f'Programação   (secao ∋ "PROG"):       {programacao.sum():>4}')

print('\n=== Cargos de gerentes ===')
print(dim_func_df[gerentes]['cargo'].value_counts().to_string())

print('\n=== Cargos de coordenadores ===')
print(dim_func_df[coordenadores]['cargo'].value_counts().to_string())

print('\n=== Valores de secao com PROG ===')
prog_vals = (
    dim_func_df[programacao]['secao']
    .str.split('|').explode().str.strip()
    .pipe(lambda s: s[s.str.contains('PROG', case=False, na=False)])
)
print(prog_vals.value_counts().to_string())

---
## 3. Construção de dim_rls_usuarios
*(células a desenvolver após mapear colunas e criar as listas SharePoint)*

Entradas:
- `dim_funcionarios` — base oficial (lotação pode estar desatualizada)
- SharePoint `rls_grupos` — grupos → escopo + perfil_acesso
- SharePoint `rls_overrides` — exceções individuais

Saída: `lake_gold_fatos.dbo.dim_rls_usuarios`

| coluna | descrição |
|--------|-----------|
| email | identificador do usuário (USERPRINCIPALNAME no Power BI) |
| escopo | UNIDADE / GERENTE_SEDE / ADMIN_CENTRAL |
| perfil_acesso | PROGRAMACAO / COORD_SUPERVISOR |
| unidade | código UO (para escopo UNIDADE) |
| gerencias | pipe-separado (para escopo GERENTE_SEDE) |
| fonte | OFICIAL / OVERRIDE — para auditoria |